In [ ]:
from datasets import load_from_disk
from gensim.corpora import Dictionary
from tokenizers import Tokenizer
from transformers import PreTrainedTokenizerFast
import yaml
import numpy as np
import torch.nn.functional as F
import torch
import torch.nn as nn

with open('./../../configs/notebooks/model04.yaml', 'r') as f:
  config = yaml.safe_load(f)

dataset = load_from_disk("../../data/processed/notebooks/ast_BPE")
code_dictionary = Dictionary.load('./../../data/processed/notebooks/ast_BPE/code_dictionary.pt')

tokenizer = PreTrainedTokenizerFast(tokenizer_file="../../data/processed/notebooks/ast_BPE/bpe_tokenizer.json",
                                    pad_token="[PAD]",
                                    bos_token="[BOS]",
                                    eos_token="[EOS]",
                                    unk_token="[UNK]")

code_dictionary.id2token = {
    v: k for k, v in code_dictionary.token2id.items()
}

c:\Users\Sean Andreini\Desktop\Unifi\Machine Learning for Software Analysis\code-summarization-mlsa-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Just a quick sanity check.

In [2]:
print(code_dictionary.token2id.keys())
print(dataset['train'][0])

dict_keys(['Add', 'Assign', 'Attribute', 'BinOp', 'Call', 'Constant', 'Expr', 'FunctionDef', 'Load', 'Module', 'Name', 'Return', 'Store', 'Subscript', 'arg', 'arguments', 'Compare', 'Eq', 'ExceptHandler', 'If', 'Not', 'Try', 'UnaryOp', 'Raise', 'And', 'Assert', 'BoolOp', 'Continue', 'Dict', 'For', 'Gt', 'In', 'Is', 'ListComp', 'NotIn', 'Tuple', 'comprehension', 'List', 'With', 'withitem', 'Break', 'Slice', 'USub', 'Mult', 'IsNot', 'AugAssign', 'Div', 'GtE', 'Lt', 'LtE', 'Mod', 'Or', 'Sub', 'While', 'Yield', 'Pass', 'keyword', 'Lambda', 'DictComp', 'IfExp', 'NotEq', 'Starred', 'GeneratorExp', 'BitAnd', 'BitOr', 'BitXor', 'RShift', 'LShift', 'Del', 'Delete', 'Invert', 'Global', 'ImportFrom', 'alias', 'Import', 'Set', 'SetComp', 'AsyncFunctionDef', 'Await', 'Pow', 'FormattedValue', 'JoinedStr', 'AnnAssign', 'FloorDiv', 'ClassDef', 'AsyncWith', 'UAdd', 'YieldFrom', 'Nonlocal', 'AsyncFor', 'MatMult', '[UNK]', '[PAD]', '[BOS]', '[EOS]'])
{'input_ids': [9, 7, 15, 14, 14, 5, 6, 5, 92, 10, 12, 

Let's now just paste the model from notebook 06.

In [3]:
from torch.nn.utils.rnn import pad_sequence
import torch

torch.cuda.empty_cache() # clears GPU memory
device = 'cuda' if torch.cuda.is_available() else 'cpu'

class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, dropout=0.2):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.embedding_dim = embedding_dim
        self.hidden = None # final hidden state
        self.cell = None
        self.basic_rnn = nn.LSTM(self.embedding_dim, self.hidden_dim, dropout=dropout, batch_first=True) # NLF

    def forward(self, X):
        embedded = self.embedding(X)
        batch_first_output, (self.hidden, self.cell) = self.basic_rnn(embedded) # NLH, 1NH, 1NH
        return batch_first_output, (self.hidden, self.cell)
    
class Attention(nn.Module):
  def __init__(self, hidden_dim, input_dim=None, proj_values=False):
    super().__init__()
    self.d_k = hidden_dim
    self.input_dim = hidden_dim if input_dim is None else input_dim
    self.proj_values = proj_values
    # Affine transformations for Q, K, and V
    self.linear_query = nn.Linear(self.input_dim, hidden_dim)
    self.linear_key = nn.Linear(self.input_dim, hidden_dim)
    self.linear_value = nn.Linear(self.input_dim, hidden_dim)
    self.alphas = None

  def init_keys(self, keys):
    self.keys = keys
    self.proj_keys = self.linear_key(self.keys)
    self.values = self.linear_value(self.keys) \
                  if self.proj_values else self.keys

  def score_function(self, query):
    proj_query = self.linear_query(query)
    # scaled dot product
    # N, 1, H x N, H, L -> N, 1, L
    dot_products = torch.bmm(proj_query, self.proj_keys.permute(0, 2, 1))
    scores =  dot_products / np.sqrt(self.d_k)
    return scores

  def forward(self, query, mask=None):
    # Query is batch-first N, 1, H
    scores = self.score_function(query) # N, 1, L
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    alphas = F.softmax(scores, dim=-1) # N, 1, L
    self.alphas = alphas.detach()

    # N, 1, L x N, L, H -> N, 1, H
    context = torch.bmm(alphas, self.values)
    return context
  
class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, bos_id, eos_id, pad_id, dropout = 0.2):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.embedding_dim = embedding_dim
        self.vocab_size = vocab_size
        self.hidden = None
        self.cell = None
        self.bos_id = bos_id
        self.eos_id = eos_id
        self.pad_id = pad_id
        self.attention = Attention(hidden_dim)
        self.basic_rnn = nn.LSTM(self.embedding_dim, self.hidden_dim, dropout=dropout, batch_first=True) # NLF
        self.output_layer = nn.Linear(2*self.hidden_dim, self.vocab_size) # 2* because of context+query

    def init_hidden(self, encoder_states):
        self.hidden, self.cell = encoder_states
        self.attention.init_keys(encoder_states[0].permute(1,0,2)) # attention wants batch first

    def forward(self, X, mask=None):
        # X is N, 1, F
        embedded = self.embedding(X)
        batch_first_output, (self.hidden, self.cell) = self.basic_rnn(embedded, (self.hidden, self.cell))
        
        # attention 
        query = batch_first_output[:, -1:, :]
        context = self.attention(query, mask=mask)
        concatenated = torch.cat([context, query], axis=-1)
        logits = self.output_layer(concatenated)
        return logits, (self.hidden, self.cell)
    
class EncoderDecoderAttn(nn.Module):
    def __init__(self, encoder, decoder, teacher_forcing_prob=0.5):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.teacher_forcing_prob = teacher_forcing_prob
        self.outputs = None
        self.alphas = None

    def init_outputs(self, batch_size, target_len):
        device = next(self.parameters()).device
        # N, L, V (output is logits)
        self.outputs = torch.zeros(
            batch_size,
            target_len,
            self.decoder.vocab_size).to(device)
        # N, L (target), L (source)
        self.alphas = torch.zeros(batch_size,
            target_len,
            self.input_len).to(device)

    def store_output(self, i, out):
        # Stores the output
        self.outputs[:, i:i+1, :] = out
        self.alphas[:, i:i+1, :] = self.decoder.attn.alphas

        
    def forward(self, source_seq, target_seq, source_mask=None):
      batch_size = source_seq.size(0)
      target_len = target_seq.size(1)
      self.input_len = source_seq.size(1)

      encoder_outputs, (enc_hidden, enc_cell) = self.encoder(source_seq)

      self.decoder.init_hidden((enc_hidden, enc_cell))
      self.decoder.attention.init_keys(encoder_outputs)
      self.init_outputs(batch_size, target_len)

      dec_inputs = torch.full((batch_size, 1), 
                              self.decoder.bos_id,
                              dtype=torch.long).to(source_seq.device)
      
      for t in range(target_seq.size(1)):
        logits, _ = self.decoder(dec_inputs, mask=source_mask)
        self.outputs[:, t, :] = logits.squeeze(1)
        self.alphas[:, t:t+1, :] = self.decoder.attention.alphas

        if self.training and torch.rand(1).item() < self.teacher_forcing_prob:
           dec_inputs = target_seq[:, t:t+1] # if teacher forcing, use actual next token as next input
        else:
           dec_inputs = logits.argmax(dim=-1) # else, use predicted token
        
      return self.outputs

In [4]:
class CollateFn:
  def __init__(self, src_pad_id, tgt_pad_id):
    self.src_pad_id = src_pad_id
    self.tgt_pad_id = tgt_pad_id

  def __call__(self, batch):
    input_ids = [torch.tensor(x['input_ids'], dtype=torch.long) for x in batch]
    labels = []
    for x in batch:
      seq = x['labels']+[tokenizer.eos_token_id]
      labels.append(torch.tensor(seq, dtype=torch.long))

    input_ids = pad_sequence(
      input_ids,
      batch_first=True,
      padding_value=self.src_pad_id
    )

    labels = pad_sequence(
      labels,
      batch_first=True,
      padding_value=self.tgt_pad_id
    )

    return input_ids, labels
  
collate = CollateFn(
  src_pad_id=code_dictionary.token2id['[PAD]'],
  tgt_pad_id=tokenizer.pad_token_id
)

In [5]:
import os

def save_checkpoint(epoch, model, optimizer, val_loss):
    os.makedirs(config['checkpoint_dir'], exist_ok=True)
    checkpoint_path = f'{config["checkpoint_dir"]}/best_model.pt'
    torch.save({'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss,
                }, checkpoint_path)
  
def load_checkpoint(model, optimizer):
    checkpoint_path = f'{config["checkpoint_dir"]}/best_model.pt'
    if(not os.path.exists(checkpoint_path)):
        print("No checkpoint found, starting from scratch")
        return 0
    
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch']
    print(f"Loaded checkpoint from epoch {start_epoch}")

    for state in optimizer.state.values():
      for k, v in state.items():
          if isinstance(v, torch.Tensor):
              state[k] = v.to(device)
    return start_epoch

In [6]:
from torch.utils.data import DataLoader
generator = torch.Generator()
generator.manual_seed(42)
torch.manual_seed(42)

train_dataset = dataset['train'].select(range(config['train_data_length']))
valid_dataset = dataset['valid'].select(range(config['valid_data_length']))

batch_size = config['batch_size']

train_dataloader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    pin_memory=True,
    collate_fn=collate,
    generator=generator
)

valid_dataloader = DataLoader(
    valid_dataset,
    batch_size=batch_size,
    shuffle=True,
    pin_memory=True,
    collate_fn=collate,
    generator=generator
)

code_vocab_size = len(code_dictionary.token2id)
docstring_vocab_size = tokenizer.vocab_size
embedding_dim = config['embedding_dim']
hidden_dim = config['hidden_dim']

encoder = Encoder(
  vocab_size=code_vocab_size,
  embedding_dim=embedding_dim,
  hidden_dim=hidden_dim
)

decoder = Decoder(
  vocab_size=docstring_vocab_size, 
  embedding_dim=embedding_dim, 
  hidden_dim=hidden_dim,
  bos_id=tokenizer.bos_token_id,
  eos_id=tokenizer.eos_token_id,
  pad_id=tokenizer.pad_token_id
)

model = EncoderDecoderAttn(
  encoder=encoder,
  decoder=decoder,
  teacher_forcing_prob=config['teacher_forcing_prob']
)



loss = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)
optimizer = torch.optim.Adam(model.parameters(), lr=config['learning_rate'])

c:\Users\Sean Andreini\Desktop\Unifi\Machine Learning for Software Analysis\code-summarization-mlsa-project\.venv\Lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(


In [7]:

epochs = config['num_epochs']
start_epoch = load_checkpoint(model, optimizer)+1

#device = 'cpu'
model.to(device)
best_loss = float('inf')
counter = 0

for epoch in range(start_epoch, epochs):
  model.train()
  batch_losses = []
  for input, labels in train_dataloader:
    input = input.to(device, non_blocking=True)
    labels = labels.to(device, non_blocking=True)
    optimizer.zero_grad()

    source_mask = (input != code_dictionary.token2id['[PAD]']).unsqueeze(1)

    y_pred = model(input, labels, source_mask=source_mask)
    y_pred = y_pred.permute(0, 2, 1)  # N, V, L
    single_loss = loss(y_pred, labels)
    single_loss.backward()
    optimizer.step()
    batch_losses.append(single_loss.item())
  
  if(epoch % 5 == 0):
    print(f"Epoch {epoch:3}, Training Loss: {sum(batch_losses)/len(batch_losses):10.8f}")

  model.eval()
  with torch.no_grad():
    val_losses = []
    for input, labels in valid_dataloader:
      input = input.to('cuda' if torch.cuda.is_available() else 'cpu', non_blocking=True)
      labels = labels.to('cuda' if torch.cuda.is_available() else 'cpu', non_blocking=True)
      y_pred = model(input, labels)
      y_pred = y_pred.permute(0, 2, 1)  # N, V, L
      single_loss = loss(y_pred, labels)
      val_losses.append(single_loss.item())

  if epoch % 5 == 0:
    print(f"Epoch {epoch:3}, Validation Loss: {sum(val_losses)/len(val_losses):10.8f}")

  if(len(val_losses) > 0 and sum(val_losses)/len(val_losses) < best_loss):
    best_loss = sum(val_losses)/len(val_losses)
    counter = 0
    save_checkpoint(epoch, model, optimizer, best_loss)
    print(f"Saved new best model (epoch {epoch})")
    
  else:
    counter += 1
    if counter >= config['patience']:
      print("Early stopping")
      break
  
  torch.cuda.empty_cache() # clears GPU memory
  del single_loss
  del y_pred
  del input
  del labels

No checkpoint found, starting from scratch
Saved new best model (epoch 1)
Epoch   5, Training Loss: 4.66362561
Epoch   5, Validation Loss: 6.84951255
Early stopping


We can see that it's actually a lot faster than the other models.

In [8]:
def predict_ids(model, input_seq, max_length=50):
  if len(input_seq)==0:
    return None
  model.eval()
  device = next(model.parameters()).device

  input_tensor = torch.tensor(input_seq, dtype=torch.long).unsqueeze(0).to(device)  # 1, L

  encoder_outputs, (enc_hidden, enc_cell) = model.encoder(input_tensor)
  model.decoder.init_hidden((enc_hidden, enc_cell))
  dec_input = torch.tensor([[model.decoder.bos_id]], dtype=torch.long).to(device)  # 1, 1

  pred_ids = []
  for _ in range(max_length):
    logits, (dec_hidden, dec_cell) = model.decoder(dec_input)

    temperature = 0.7
    probs = torch.softmax(logits / temperature, dim=-1)
    next_token_id = torch.multinomial(probs.squeeze(0), 1).item()

    if(next_token_id == model.decoder.eos_id):
      break
    pred_ids.append(next_token_id)
    dec_input = torch.tensor([[next_token_id]], dtype=torch.long).to(device)  # 1, 1
  
  return pred_ids
  
def decode_ids(pred_ids):
  return [tokenizer.decode(token_id) for token_id in pred_ids]

test_dataset = dataset['test'].select(range(config['test_data_length']))

test_dataloader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    pin_memory=True,
    collate_fn=collate
)

In [9]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

smooth_fn = SmoothingFunction().method4
bleu_scores = []
rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)
rouge_scores = []

test_dataset = dataset['test'].select(range(config['test_data_length']))

load_checkpoint(model, optimizer)
model.eval()
model.to(device)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    pin_memory=True,
    collate_fn=collate
)


for(input, labels) in test_dataloader:
  for i in range(input.size(0)):
    with torch.no_grad():
      pred_ids = predict_ids(model, input[i].tolist())
    
    if pred_ids == None:
       continue
    pred_tokens  = decode_ids(pred_ids)
    labels_tokens = decode_ids(labels[i].tolist())

    print("Predicted Docstring: ", ' '.join(pred_tokens))
    print("Actual Docstring:    ", ' '.join(labels_tokens))

    bleu_score = sentence_bleu(
        [labels_tokens],
        pred_tokens,
        smoothing_function=smooth_fn
    )
    rouge_score = rouge.score(' '.join(labels_tokens), ' '.join(pred_tokens))['rougeL'].fmeasure

    bleu_scores.append(bleu_score)
    rouge_scores.append(rouge_score)

print(f"Average BLEU score on test set: {sum(bleu_scores)/len(bleu_scores):.4f}")
print(f"Average RougeL score on test set: {sum(rouge_scores)/len(rouge_scores):.4f}")

Loaded checkpoint from epoch 1
Predicted Docstring:  returns to the
Actual Docstring:     str - > list convert serial to cut list . from notification li coverage . [EOS]
Predicted Docstring:  runs for and represented the ing dictionary
Actual Docstring:     revis avatar interpolate remin by cut . [EOS]
Predicted Docstring:  check ance to values
Actual Docstring:     revis get_ a remin by cut . [EOS]
Predicted Docstring:  a to file
Actual Docstring:     format whe with than or other zen gen _state vation string . [EOS]
Predicted Docstring:  add a umb sto . .
Actual Docstring:     down a ans uct to stack default . [EOS]
Predicted Docstring:  check a new from the before is a connect and _ graph .
Actual Docstring:     down an default ans uct . [EOS]
Predicted Docstring:  ant a up to a new the
Actual Docstring:     frequ a ter ri ble _list ! [EOS]
Predicted Docstring:  returns the install top .
Actual Docstring:     termin benchmark place . [EOS]
Predicted Docstring:  returns a provid from